### FIFA Network Analysis

Builds a mention graph from the raw working data, detects communities (Louvain), joins per-tweet emotion labels to compute per-author dominant emotion, and produces community-level emotion statistics for Experiment 3 (Network Emotional Homogeneity).

**Inputs**
- `FIFA Working Data.csv` (raw, 1.5M rows) — `ID`, `Name`, `UserMentionNames`, `Date`
- `fifa_tweets_sentiment.csv` (cleaned, 113,821 rows) — `ID`, `emotion_label`, `nearest_event_match`

**Outputs (`static/data/network/`)**
- `community_stats.csv` — per-community size, dominant emotion, entropy
- `nodes.json` — top-community node metadata for dashboard
- `edges.json` — corresponding edges
- `homogeneity.json` — within vs across-community emotion-agreement summary

In [1]:
from pathlib import Path
import json, re, math
from collections import Counter, defaultdict
import pandas as pd
import networkx as nx
import community as community_louvain

REPO_ROOT = Path('..').resolve()
WORKING_CSV = Path('/Users/williamyoo/Downloads/drive-download-20260423T200046Z-3-001/FIFA Working Data.csv')
SENTIMENT_CSV = REPO_ROOT / 'static' / 'data' / 'fifa_tweets_sentiment.csv'
OUT_DIR = REPO_ROOT / 'static' / 'data' / 'network'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('working:', WORKING_CSV.exists(), WORKING_CSV)
print('sentiment:', SENTIMENT_CSV.exists(), SENTIMENT_CSV)
print('out dir:', OUT_DIR)

working: True /Users/williamyoo/Downloads/drive-download-20260423T200046Z-3-001/FIFA Working Data.csv
sentiment: True /Users/williamyoo/Documents/GitHub/cse-6242-tm251/static/data/fifa_tweets_sentiment.csv
out dir: /Users/williamyoo/Documents/GitHub/cse-6242-tm251/static/data/network


In [2]:
sent = pd.read_csv(SENTIMENT_CSV, usecols=['ID', 'emotion_label', 'nearest_event_match', 'datetime'])
print('sentiment rows:', len(sent))
print(sent['emotion_label'].value_counts(normalize=True).round(3))
sent.head(3)

sentiment rows: 113821
emotion_label
neutral     0.410
joy         0.247
surprise    0.107
sadness     0.105
anger       0.077
fear        0.046
disgust     0.009
Name: proportion, dtype: float64


,ID,datetime,nearest_event_match,emotion_label
0,1013597060640145408,2018-07-02 01:35:45,none,anger
1,1013597056219295744,2018-07-02 01:35:44,none,joy
2,1013597047482544130,2018-07-02 01:35:42,none,neutral


In [3]:
work = pd.read_csv(
    WORKING_CSV,
    usecols=['ID', 'Name', 'UserMentionNames', 'Date'],
    low_memory=False,
    encoding='latin-1',
)
work = work.dropna(subset=['Name'])
print('working rows:', len(work))
print('unique authors:', work['Name'].nunique())
print('rows with mentions:', work['UserMentionNames'].notna().sum())

working rows: 529944
unique authors: 288297
rows with mentions: 455786


In [4]:
emo = work[['ID', 'Name']].merge(sent[['ID', 'emotion_label', 'nearest_event_match']], on='ID', how='inner')
print('labeled rows:', len(emo))

author_emo_counts = emo.groupby(['Name', 'emotion_label']).size().unstack(fill_value=0)
author_dominant = author_emo_counts.idxmax(axis=1).rename('dominant_emotion')
author_tweet_count = author_emo_counts.sum(axis=1).rename('tweet_count')
author_df = pd.concat([author_dominant, author_tweet_count], axis=1).reset_index()
author_df.head(10)

labeled rows: 113813


,Name,dominant_emotion,tweet_count
0,!,sadness,1
1,! AWESOMAZING Â¡,surprise,2
2,!! ????? ?????? !!,fear,1
3,!!!,fear,1
4,!?,joy,1
5,!NTZ,joy,1
6,!mR@n,anger,2
7,"""????? ??????"" ver.??????????????",anger,1
8,"""Dont Give A Damn"". ?",fear,1
9,"""The Process""",neutral,1


In [5]:
mention_rows = work.dropna(subset=['UserMentionNames']).copy()
mention_rows = mention_rows[mention_rows['UserMentionNames'].astype(str).str.len() > 0]

# Only keep interactions where BOTH source and target authored a labeled tweet in our sentiment dataset.
# This guarantees every graph node has a real emotion label.
labeled_authors = set(author_df['Name'])
print('labeled authors available as nodes:', len(labeled_authors))

edge_counter = Counter()
for src, ments in zip(mention_rows['Name'].astype(str), mention_rows['UserMentionNames'].astype(str)):
    if src not in labeled_authors:
        continue
    for tgt in [m.strip() for m in ments.split(',') if m.strip()]:
        if tgt and tgt != src and tgt in labeled_authors:
            edge_counter[(src, tgt)] += 1

print('labeled-to-labeled directed edges:', len(edge_counter))
print('total mentions in labeled subgraph:', sum(edge_counter.values()))

labeled authors available as nodes: 77105


labeled-to-labeled directed edges: 69786
total mentions in labeled subgraph: 123318


In [6]:
MIN_EDGE_WEIGHT = 1
filtered_edges = [(s, t, w) for (s, t), w in edge_counter.items() if w >= MIN_EDGE_WEIGHT]
print(f'edges after weight>={MIN_EDGE_WEIGHT}:', len(filtered_edges))

G = nx.DiGraph()
for s, t, w in filtered_edges:
    G.add_edge(s, t, weight=w)
print('nodes:', G.number_of_nodes(), 'edges:', G.number_of_edges())

UG = G.to_undirected()
for u, v, d in UG.edges(data=True):
    UG[u][v]['weight'] = G[u][v]['weight'] if G.has_edge(u, v) else G[v][u]['weight']

components = sorted(nx.connected_components(UG), key=len, reverse=True)
print('num components:', len(components), '| largest:', len(components[0]))
LCC = UG.subgraph(components[0]).copy()
print('LCC nodes:', LCC.number_of_nodes(), 'edges:', LCC.number_of_edges())

edges after weight>=1: 69786
nodes: 33531 edges: 69786


num components: 1854 | largest: 29389


LCC nodes: 29389 edges: 66912


In [7]:
partition = community_louvain.best_partition(LCC, weight='weight', random_state=42)
num_communities = len(set(partition.values()))
modularity = community_louvain.modularity(partition, LCC, weight='weight')
print('communities:', num_communities, '| modularity:', round(modularity, 4))

comm_sizes = Counter(partition.values())
top_comms = comm_sizes.most_common(15)
print('top 15 community sizes:', top_comms)

communities: 119 | modularity: 0.5018
top 15 community sizes: [(7, 5014), (6, 4888), (2, 2907), (0, 2408), (1, 1940), (24, 1403), (11, 1189), (26, 1097), (36, 824), (14, 691), (43, 689), (3, 682), (5, 677), (20, 629), (50, 563)]


In [8]:
author_emo_lookup = dict(zip(author_df['Name'], author_df['dominant_emotion']))
author_count_lookup = dict(zip(author_df['Name'], author_df['tweet_count']))

EMO_LABELS = ['joy', 'sadness', 'anger', 'fear', 'disgust', 'surprise', 'neutral']

def entropy(counts_dict):
    total = sum(counts_dict.values())
    if total == 0:
        return 0.0
    return -sum((c/total) * math.log2(c/total) for c in counts_dict.values() if c > 0)

comm_rows = []
for cid, size in comm_sizes.items():
    members = [n for n, c in partition.items() if c == cid]
    emo_counts = Counter(author_emo_lookup.get(n, 'unknown') for n in members)
    tweets = sum(author_count_lookup.get(n, 0) for n in members)
    dominant = emo_counts.most_common(1)[0][0] if emo_counts else 'unknown'
    agree = emo_counts.most_common(1)[0][1] / size if size > 0 else 0.0
    ent = entropy({k: v for k, v in emo_counts.items() if k in EMO_LABELS})
    comm_rows.append({
        'community_id': cid,
        'size': size,
        'tweet_count': tweets,
        'dominant_emotion': dominant,
        'dominant_agreement': round(agree, 4),
        'entropy_bits': round(ent, 4),
        **{f'n_{e}': emo_counts.get(e, 0) for e in EMO_LABELS},
    })
comm_df = pd.DataFrame(comm_rows).sort_values('size', ascending=False).reset_index(drop=True)
comm_df.head(15)

,community_id,size,tweet_count,dominant_emotion,dominant_agreement,entropy_bits,n_joy,n_sadness,n_anger,n_fear,n_disgust,n_surprise,n_neutral
0,7,5014,9476,neutral,0.3785,2.2520,1496,444,440,263,39,434,1898
1,6,4888,9591,neutral,0.4178,2.1678,1434,400,374,215,41,382,2042
2,2,2907,6748,neutral,0.5067,1.9755,759,163,188,96,20,208,1473
3,0,2408,4175,neutral,0.4236,2.1989,662,195,203,129,24,175,1020
4,1,1940,3425,neutral,0.3995,2.2325,542,191,181,97,14,140,775
5,24,1403,2398,neutral,0.3885,2.2860,380,135,128,77,14,124,545
6,11,1189,2000,neutral,0.4449,2.1418,320,76,83,68,7,106,529
7,26,1097,1833,neutral,0.3747,2.2671,320,79,131,59,10,87,411
8,36,824,1351,neutral,0.4223,2.2041,225,68,64,43,8,68,348
9,14,691,1268,joy,0.3488,2.2643,241,52,85,34,4,58,217


In [9]:
# Within- vs across-community emotion agreement
# Within: mean dominant_agreement weighted by community size, over communities with size>=10 and labeled members
min_size = 10
sig = comm_df[(comm_df['size'] >= min_size) & (comm_df['dominant_emotion'].isin(EMO_LABELS))]
within = (sig['dominant_agreement'] * sig['size']).sum() / sig['size'].sum() if sig['size'].sum() > 0 else 0.0

# Across baseline: pool all labeled authors, what fraction share the most common emotion?
all_labeled = [author_emo_lookup.get(n) for n in partition if author_emo_lookup.get(n) in EMO_LABELS]
global_counts = Counter(all_labeled)
across = global_counts.most_common(1)[0][1] / sum(global_counts.values()) if global_counts else 0.0

# Assortativity on emotion label
for n in LCC.nodes():
    LCC.nodes[n]['emo'] = author_emo_lookup.get(n, 'unknown')
try:
    assort = nx.attribute_assortativity_coefficient(LCC, 'emo')
except Exception as e:
    assort = float('nan')
    print('assortativity failed:', e)

homogeneity = {
    'within_community_agreement': round(within, 4),
    'across_community_baseline': round(across, 4),
    'lift': round(within - across, 4),
    'emotion_assortativity': round(assort, 4) if not math.isnan(assort) else None,
    'num_communities_considered': int(len(sig)),
    'modularity': round(modularity, 4),
    'total_nodes': LCC.number_of_nodes(),
    'total_edges': LCC.number_of_edges(),
    'min_community_size': min_size,
}
homogeneity

{'within_community_agreement': np.float64(0.4146),
 'across_community_baseline': 0.413,
 'lift': np.float64(0.0017),
 'emotion_assortativity': 0.027,
 'num_communities_considered': 39,
 'modularity': 0.5018,
 'total_nodes': 29389,
 'total_edges': 66912,
 'min_community_size': 10}

In [10]:
TOP_N_COMMUNITIES = 8
top_community_ids = [cid for cid, _ in comm_sizes.most_common(TOP_N_COMMUNITIES)]
top_nodes = {n for n, c in partition.items() if c in top_community_ids}

MAX_NODES = 400
if len(top_nodes) > MAX_NODES:
    degree = dict(LCC.degree(weight='weight'))
    top_nodes = set(sorted(top_nodes, key=lambda n: degree.get(n, 0), reverse=True)[:MAX_NODES])

sub = LCC.subgraph(top_nodes).copy()

nodes_json = []
for n in sub.nodes():
    nodes_json.append({
        'id': n,
        'community': int(partition[n]),
        'emotion': author_emo_lookup.get(n, 'unknown'),
        'tweets': int(author_count_lookup.get(n, 0)),
        'degree': int(sub.degree(n)),
    })
edges_json = [{'source': u, 'target': v, 'weight': int(d.get('weight', 1))} for u, v, d in sub.edges(data=True)]
print('exported nodes:', len(nodes_json), 'edges:', len(edges_json))

exported nodes: 400 edges: 4221


In [11]:
comm_df.to_csv(OUT_DIR / 'community_stats.csv', index=False)
with open(OUT_DIR / 'nodes.json', 'w') as f:
    json.dump(nodes_json, f)
with open(OUT_DIR / 'edges.json', 'w') as f:
    json.dump(edges_json, f)
with open(OUT_DIR / 'homogeneity.json', 'w') as f:
    json.dump(homogeneity, f, indent=2)
print('wrote outputs to', OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print(' ', p.name, p.stat().st_size, 'bytes')

wrote outputs to /Users/williamyoo/Documents/GitHub/cse-6242-tm251/static/data/network
  community_stats.csv 5278 bytes
  edges.json 263908 bytes
  homogeneity.json 268 bytes
  nodes.json 34791 bytes


### Notes and known limitations

- Graph is built from `Name` (display name) + `UserMentionNames` (comma-separated display names). Retweets are a subset of mentions, so the mention graph subsumes the RT graph while keeping node identity consistent (handles would mix with display names).
- Nodes without mentions and authors of only untagged tweets are excluded. The graph captures public interaction structure, not the full user base.
- Display names are not unique on Twitter; some conflation is possible but unlikely to dominate the community structure given the filter MIN_EDGE_WEIGHT>=2.
- Community detection (Louvain) is run on the undirected projection with edge weights = mention count, to match standard practice and stabilize partitions.